# *Table of contents*
>## 1. [Understanding Diffusion Models: A Unified Perspective](#understanding-diffusion-models-a-unified-perspective)
>>### 1. [Generative Models](#generative-models)
>>### 2. [Background](#background)
>>### 3. [ELBO](#elbo)
>>### 4. [VAE](#vae)
>>### 5. [HVAE](#hvae)
>>### 6. [VDM](#vdm)


[To document tail](#document-tail)
***
***
***
***

# [**Understanding Diffusion Models: A Unified Perspective**](#1-understanding-diffusion-models-a-unified-perspective)
---
---
---

# [*Generative Models*](#table-of-contents)
---
---

표본 x가 주어진 상태
- 목표: $p(x)$ 배우기 ($p(x)$ = true data distribution)
- $p(x)$ 배운 모델 $p_{\theta}(x)$에서 sampling하면 생성
    - $p_{\theta}(x)$를 이용해 $x_{관측}$ 또는 $x_{생성}$의 우도(likelihood)도 계산 가능

## **생성 모델 분류**
- GAN
- likelihood-based
    - VAE
- energy-based
    - score-based

---
---

# [*Background*](#table-of-contents)
---
---

[베이즈 룰 이해](https://hyeongminlee.github.io/post/bnn001_bayes_rule/)  

![Bayes' rule](imgs/Bayes.png)  
[그림 출처](https://blog.naver.com/ycpiglet/223058716063)

---

![latent_z](imgs/latent_z.png)

![plato_vs_gen ](imgs/plato_vs_generator.png)

---
---

# [*ELBO*](#table-of-contents)
---
---

![posterior](imgs/bayes_posterior_evidence.png)

## **기본 세팅**
"likelihood-based"계열 생성모델은 모든 관측값 x의 likelihood $p(x)$ (엄밀히는 marginal likelihood 또는 evidence)를 최대화하는 방향으로 학습을 한다. latent variable $z$와 observed data $x$를 joint distribution $p(x,z)$로 모델링할 경우, Equation 1, Equation 2를 통해 $p(x)$를 얻을 수 있다.


**Equation 1:**
$$p(\mathbf{x}) = \int p(\mathbf{x}, \mathbf{z})d\mathbf{z}$$

**Equation 2 (확률의 연쇄 법칙):**
$$p(\mathbf{x}) = \frac{p(\mathbf{x}, \mathbf{z})}{p(\mathbf{z}|\mathbf{x})}$$

그러나 $p(x)$를 직접 계산하는건 
- Equation1에서 모든 latent variable $z$에 대해 적분하거나
    - 복잡한 모델의 경우 intractable
- Equation2에서 ground truth latent encoder $p(\mathbf{z}|\mathbf{x})$를 알아야 해서  

매우 어렵다.

=> evidence $p(x)$를 직접 계산하는 대신 Evidence의 Lower BOund 즉 ELBO를 Eq. 1, Eq. 2 에서 얻어내어 간접적으로 접근한다.

**Equation 3 (ELBO):**
$$\mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log\frac{p(\mathbf{x}, \mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\right]$$

여기서 $q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})$는 ground truth encoder $p(\mathbf{z}|\mathbf{x})$를 모수(parameter) $\phi$를 조정해 근사하는 모델로 $q(\mathbf{z}|\mathbf{x}, \boldsymbol{\phi})$로도 표현가능

**Equation 4:**
$$\log p(\mathbf{x}) \geq \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log\frac{p(\mathbf{x}, \mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\right]$$

ELBO는 evidence의 하한이므로 ELBO를 대신 구해 최대화한다.

## **ELBO 유도 1**

**Equation 5~8:**
$$\begin{aligned} \log p(\mathbf{x}) & = \log \int p(\mathbf{x}, \mathbf{z})d\mathbf{z} \qquad (Eq.\ 1)\\ & = \log \int \frac{p(\mathbf{x}, \mathbf{z})q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}d\mathbf{z} \qquad (\text{Multiply\ by\ }1\ = \frac{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}) \\ & = \log \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\frac{p(\mathbf{x}, \mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\right] \qquad (\text{Def\ of\ Expectation\ for\ }\mathbf{z} \sim q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})) \\ & \geq \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log \frac{p(\mathbf{x}, \mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\right] \qquad (\text{Jensen's\ Inequality}) \end{aligned}$$

이 유도방식은 Jensen's Inequality를 이용해 ELBO가 하한인 것을 보이기는 했으나 log eveidence와 ELBO가 정확하게 어떻게 엮이는지에 대해 알기에는 부족하다.

## **ELBO 유도 2**

**Equation 9~16:**
$$\begin{aligned} \log p(\mathbf{x}) & = \log p(\mathbf{x}) \int q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})d\mathbf{z} \qquad (\text{Multiply\ by\ }1\ =\int q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})d\mathbf{z}) \\ & = \int q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})(\log p(\mathbf{x}))d\mathbf{z} \\ & = \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log p(\mathbf{x})\right] \\ & = \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log\frac{p(\mathbf{x}, \mathbf{z})}{p(\mathbf{z}|\mathbf{x})}\right] \qquad (\text{Eq.\ 2}) \\ & = \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log\frac{p(\mathbf{x}, \mathbf{z})q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}{p(\mathbf{z}|\mathbf{x})q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\right] \qquad (\text{Multiply\ by\ 1\ }= \frac{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}) \\ & = \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log\frac{p(\mathbf{x}, \mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\right] + \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log\frac{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}{p(\mathbf{z}|\mathbf{x})}\right] \\ & = \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log\frac{p(\mathbf{x}, \mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\right] + D_{\text{KL}}(q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}|\mathbf{x})) \qquad (\text{Def\ of\ KL\ Divergence}) \\ & \geq \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log\frac{p(\mathbf{x}, \mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\right] \qquad (\text{KL\ Divergence\ always }\geq 0) \end{aligned}$$

### $\therefore$ log evidence = ELBO + KL div. between the approximate posterior $q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})$ and the true posterior $p(\mathbf{z}|\mathbf{x})$
#### **log evidence - ELBO는 항상 0보다 크거나 같은 KL div. 이므로 ELBO가 항상 log evidence의 하한임을 확인 가능**

$\boldsymbol{\phi}$를 바꿔가며 $q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})$가 $p(\mathbf{z}|\mathbf{x})$에 근사하는 것이 현재 목표.

이 목표를 수식으로 표현하면 $D_{\text{KL}}(q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}|\mathbf{x})) \rightarrow 0$ 이 된다. 

이 KL div. 를 직접 $\phi$에 대한 함수로 계산하고 최소화하는건 $p(\mathbf{z}|\mathbf{x})$를 모르므로 불가능한 상황

여기서 log evidence가 $\phi$와 무관하므로( $p(x)$는 $p(x,z)$에서 모든 z를 주변화(marginalize)해서 제거한 상태 ) $\phi$에 대해 상수취급이 가능

= > 상수 = ELBO + KL div. 인 상황이므로 ELBO를 증가시키면 KL div. 부분이 줄어든다 

#### **ELBO를 증가시키면 자연스럽게 모델이 true posterior를 근사하게 된다.**

---
---

# [*VAE*](#table-of-contents)
---
---

![vae](imgs/vae.png)

$\phi$로 매개변수화된 가능한 모든 posterior의 family 에서 최적화를(= ELBO 최대화) 통해 true posterior를 잘 근사하는(= $D_{\text{KL}}(q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}|\mathbf{x}))$를 최소화하는) 가장 좋은 $q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})$ 를 구하므로 variational이라고 한다.

ELBO 항을 더 분해해보면 decoder $p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})$가 포함된 reconstruction term과 encoder $q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})$가 포함된  prior matching term 으로 분해된다

**Equation 17~19:**
$$\begin{aligned} \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log\frac{p(\mathbf{x}, \mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\right] & = \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log\frac{p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})p(\mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\right] \qquad (\text{Chain\ Rule\ of\ Probability}) \\ & = \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})\right] + \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log \frac{p(\mathbf{z})}{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\right] \\ & = \underbrace{\mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})\right]}_\text{reconstruction term} - \underbrace{D_{\text{KL}}(q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}))}_\text{prior matching term}  \qquad (\text{Def\ of\ KL\ Div.}) \end{aligned}$$

> **※** $p(\mathbf{x}, \mathbf{z}) = p(\mathbf{x}|\mathbf{z})p(\mathbf{z})$ 를 적용할 떄 true decoder $p(\mathbf{x}|\mathbf{z})$를 파라메터 $\theta$를 포함하는 학습가능한 신경망 모델 $p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})$로 대체해 $p(\mathbf{x}, \mathbf{z}) = p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})p(\mathbf{z})$ 사용

encoder $q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})$를 posterior로 보면, decoder $p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})$는 likelihood, $p(\mathbf{z})$는 prior가 된다

**Reconstruction term**
- variational dist. 기준으로 log likelihood를 계산
- $p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})$가 베르누이 분포를 따른다 가정하면 cross entropy가 되고 가우시안 분포를 따른다 가정하면 MSE로 변환된다

**prior matching term**
- 학습한 variational dist.가 prior와 얼마나 유사한가
- VAE에서는 prior를 표준정규분포로 설정해 추가적인 파라메터 의존성을 도입하지 않는다
    - $p(\mathbf{z})$를 미리 원하는 구조(표준정규분포)로 정해두면 x를 z로 압축하는 $q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})$가 정해둔 다루기 쉬운 형태로 mapping하도록 학습을 강제하는 효과가 있다.

**Equation 20, 21 (prior 가우시안 분포 가정):**
$$q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x}) = \mathcal{N}(\mathbf{z}; \boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x}), \boldsymbol{\sigma}_{\boldsymbol{\phi}}^2(\mathbf{x})\mathbf{I})$$
$$p(\mathbf{z}) = \mathcal{N}(\mathbf{z}; \mathbf{0}, \mathbf{I})$$

학습을 위해서 ELBO를 최대화 해야 하므로 reconstruction term을 최대화하고 prior matching term을 최소화하는 것이 목표가 된다.

**Equation 22 (VAE 학습 목표):**

$$\arg\max_{\boldsymbol{\phi}, \boldsymbol{\theta}} \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})}\left[\log p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z})\right] - D_{\text{KL}}(q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z})) \approx \arg\max_{\boldsymbol{\phi}, \boldsymbol{\theta}} \frac{1}{L}\sum_{l=1}^{L}\log p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z}^{(l)}) - D_{\text{KL}}(q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}))$$

prior matching term은 식 20,21의 설정을 통해([가우시안 분포 사이의 KL Div.](https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence#Multivariate_normal_distributions)) 바로 계산 가능하고, reconstruction term은 Monte Carlo estimate(sampling)를 사용해 근사한다
- 실제 학습 시에는 각 step 관측 데이터 $x$를 인코더 신경망 $q_\phi$에 통과시켜 잠재 공간(latent space)에서의 확률 분포 파라미터(일반적으로 가우시안 분포의 평균 $\mu$와 분산 $\sigma^2$)를 출력
- 출력된 파라미터로 만들어지는 확률 분포 $q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})$에서 $z$값을 무작위로 $L$번 추출. $z^{(1)}, z^{(2)}, \dots, z^{(L)}$
- 무작위로 뽑힌 $L$개의 $z$들을 각각 디코더 신경망 $p_{\boldsymbol{\theta}}$에 입력하여 원본 $x$가 나올 확률(또는 복원 오차), 즉 $\log p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z}^{(l)})$를 계산
- $L$개의 샘플에 대해 구한 복원 확률들을 모두 더한 뒤 $L$로 나누어 평균 ($\frac{1}{L}\sum$).

> **※** 실제 코드 구현시에는 $L=1$을 사용해 각 step 관측데이터 $x$ 하나당 $z$를 하나만 뽑아서 reconstruction term을 계산하고 가중치를 업데이트 한다.
>> 각 x마다 $L$번씩 $z$를 추출해서 디코더에 입력해서 확률을 계산하면 연산량이 너무 크고, 어차피 epoch을 돌면서 확률적 경사 하강법을 사용해 수많은 미니배치 데이터를 반복적으로 학습하기 때문에, 스텝마다 $z$를 1번씩만 무작위로 뽑아도 전체 학습 과정을 통틀어 보면 기댓값에 충분히 잘 수렴(근사)한다.

**Reparameterization trick**

reconstruction term 계산을 할 때, 가우시안 분포의 평균 $\mu$와 분산 $\sigma^2$를 이용해 $z$를 추출할 경우 gradient descent를 진행할 때, $z$가 미분불가능하다는 문제가 있다.
하지만 확률 분포 $q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})$가 multivariate Gaussian과 같은 좋은 분포를 따르면 reparameterization trick을 사용해 해결 가능

$x$가 임의의 평균 $\mu$, 분산 $\sigma^2$을 가지는 정규분포에서 추출한 샘플일 경우 ($x \sim\ \mathcal{N}(\mathbf{x}; \mu, \sigma^2)$)


**Equation (재매개변수화 트릭):**  
$$x = \mu + \sigma \epsilon \ with\ \epsilon \sim\ \mathcal{N}(\mathbf{z}; \mathbf{0}, \mathbf{I})$$



로 $x$를 다시 써서 $x$의 정규분포가 표준 정규분포를 평균 $\mu$, 분산 $\sigma^2$로 이동시킨거라 해석하면 무작위성을 표준 정규분포에 다 몰아넣고, $x$를 미분할 때, $\mu$와 $\sigma$ 부분이 미분 가능해진다.

이 트릭을 $z \sim q_{\boldsymbol{\phi}}(\mathbf{z}|\mathbf{x})$ 에 적용하면

**Equation (재매개변수화 트릭 적용):**
$$\mathbf{z} = \boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x}) + \boldsymbol{\sigma}_{\boldsymbol{\phi}}(\mathbf{x})\odot\boldsymbol{\epsilon} \quad \text{with } \boldsymbol{\epsilon} \sim \mathcal{N}(\boldsymbol{\epsilon};\mathbf{0}, \mathbf{I})$$

로 표현되고, 미분 불가능한 노이즈 부분 $\epsilon$과 다르게 deterministic funtion인 $\boldsymbol{\mu}_{\boldsymbol{\phi}}(\mathbf{x})$와 $\boldsymbol{\sigma}_{\boldsymbol{\phi}}(\mathbf{x})$ 부분만 $\phi$에 dependent 하므로 gradient descent 가능

![reparameterization](imgs/repara_trick.png)

[이미지 출처](https://heygeronimo.tistory.com/40)

---
---

# [*HVAE*](#table-of-contents)
---
---

## **Hierarchical Variational Autoencoders**

![hvae](imgs/hvae_m.png)

[이미지 출처](https://arxiv.org/abs/2208.11970)

VAE의 1단계 압축, 복원을 그림처럼 T단계에 걸쳐서 진행하는 것이 HVAE. 논문에서는 각 latent 변수들의 조건부확률에 바로 전단계 변수만 들어가는 Markovian HVAE (MHVAE)만을 다룬다.  


## **HVAE ELBO 유도**


VAE에서처럼 ELBO를 구하기 위해 latent variables $z_1$, ... $z_T$ 와 observed data $x$를 joint distribution $p(\mathbf{x}, \mathbf{z}_{1:T})$로 모델링한다.

> $p(\mathbf{x}, \mathbf{z}_{1:T}) = p(\mathbf{x}, \mathbf{z}_{1}, \mathbf{z}_{2}, \dots ,\mathbf{z}_{T})$

**Equation 23:**  

$$
\begin{aligned}
p(\mathbf{x}, \mathbf{z}_{1:T}) &= \frac{p(\mathbf{x}, \mathbf{z}_{1:T})}{p(\mathbf{z}_{1:T})} \frac{p(\mathbf{z}_{1:T})}{p(\mathbf{z}_{2:T})} \frac{p(\mathbf{z}_{2:T})}{p(\mathbf{z}_{3:T})}\dots \frac{p(\mathbf{z}_{T-1}, \mathbf{z}_{T})}{p(\mathbf{z}_{T})}\cdot p(\mathbf{z}_{T}) \qquad (\text{Chain\ Rule\ of\ Probability}) \\ &= p(\mathbf{x}|\mathbf{z}_{1:T}) p(\mathbf{z_1}|\mathbf{z}_{2:T}) p(\mathbf{z_2}|\mathbf{z}_{3:T}) \dots p(\mathbf{z_{T-1}}|\mathbf{z}_{T})p(\mathbf{z}_{T}) \\ &= p(\mathbf{x}|\mathbf{z}_{1}) p(\mathbf{z_1}|\mathbf{z}_{2}) p(\mathbf{z_2}|\mathbf{z}_{3}) \dots p(\mathbf{z_{T-1}}|\mathbf{z}_{T})p(\mathbf{z}_{T}) \qquad (\text{Markovian process}) \\ &= p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z}_{1}) p_{\boldsymbol{\theta}}(\mathbf{z_1}|\mathbf{z}_{2}) p_{\boldsymbol{\theta}}(\mathbf{z_2}|\mathbf{z}_{3}) \dots p_{\boldsymbol{\theta}}(\mathbf{z_{T-1}}|\mathbf{z}_{T})p(\mathbf{z}_{T}) \qquad (\theta\ \text{의존성 주입}) \\ &= p(\mathbf{z}_T)p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z}_1)\prod_{t=2}^{T}p_{\boldsymbol{\theta}}(\mathbf{z}_{t-1}|\mathbf{z}_{t}) 
\end{aligned}
$$

> true posterior $q(\mathbf{z}_{1:T}|\mathbf{x})$도 $\phi$ 파라메터 신경망 모델 $q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})$로 근사

**Equation 24:**  

$$
\begin{aligned}
q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x}) &= \frac{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}, \mathbf{x})}{p(\mathbf{x})} \\ &= \frac{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}, \mathbf{x})}{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T-1}, \mathbf{x})} \frac{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T-1}, \mathbf{x})}{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T-2}, \mathbf{x})} \dots \frac{q_{\boldsymbol{\phi}}(\mathbf{z}_{2}, \mathbf{z}_{1}, \mathbf{x})}{q_{\boldsymbol{\phi}}(\mathbf{z}_{1}, \mathbf{x})} \frac{q_{\boldsymbol{\phi}}(\mathbf{z}_{1}, \mathbf{x})}{p(\mathbf{x})} \\ &= q_{\boldsymbol{\phi}}(\mathbf{z}_T | \mathbf{z}_{1:T-1}, \mathbf{x}) q_{\boldsymbol{\phi}}(\mathbf{z}_{T-1} | \mathbf{z}_{1:T-2}, \mathbf{x}) \dots q_{\boldsymbol{\phi}}(\mathbf{z}_2 | \mathbf{z}_{1}, \mathbf{x}) q_{\boldsymbol{\phi}}(\mathbf{z}_1 | \mathbf{x}) \\ &= q_{\boldsymbol{\phi}}(\mathbf{z}_T | \mathbf{z}_{T-1}) q_{\boldsymbol{\phi}}(\mathbf{z}_{T-1} | \mathbf{z}_{T-2}) \dots q_{\boldsymbol{\phi}}(\mathbf{z}_2 | \mathbf{z}_{1}) q_{\boldsymbol{\phi}}(\mathbf{z}_1 | \mathbf{x}) \qquad (\text{Markovian process}) \\ &= q_{\boldsymbol{\phi}}(\mathbf{z}_1|\mathbf{x})\prod_{t=2}^{T}q_{\boldsymbol{\phi}}(\mathbf{z}_{t}|\mathbf{z}_{t-1})
\end{aligned}
$$

이제 VAE의 ELBO 유도 방식 1번을 그대로 적용해 MHVAE의 ELBO를 유도해보면

**Equation 25~28:**  

$$
\begin{aligned}
\log p(\mathbf{x}) &= \log \int p(\mathbf{x}, \mathbf{z}_{1:T}) d\mathbf{z}_{1:T} \qquad \text{(Eq. 1을}\ \mathbf{z}_{1:T}\ \text{로 확장)}\\
&= \log \int \frac{p(\mathbf{x}, \mathbf{z}_{1:T})q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})}{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})} d\mathbf{z}_{1:T} \qquad \text{(Multiply by } 1 = \frac{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})}{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})} \text{)}\\
&= \log \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})}\left[\frac{p(\mathbf{x}, \mathbf{z}_{1:T})}{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})}\right] \qquad \text{(Def of Expectation)}\\
&\geq \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})}\left[\log \frac{p(\mathbf{x}, \mathbf{z}_{1:T})}{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})}\right] \qquad \text{(Jensen's Inequality)}
\end{aligned}
$$

이 나오고 여기서 Eq. 23과 24를 대입하면

**Equation 29:**  

$$
\begin{aligned}
\mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})}\left[\log \frac{p(\mathbf{x}, \mathbf{z}_{1:T})}{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})}\right]
&= \mathbb{E}_{q_{\boldsymbol{\phi}}(\mathbf{z}_{1:T}|\mathbf{x})}\left[\log \frac{p(\mathbf{z}_T)p_{\boldsymbol{\theta}}(\mathbf{x}|\mathbf{z}_1)\prod_{t=2}^{T}p_{\boldsymbol{\theta}}(\mathbf{z}_{t-1}|\mathbf{z}_{t})}{q_{\boldsymbol{\phi}}(\mathbf{z}_1|\mathbf{x})\prod_{t=2}^{T}q_{\boldsymbol{\phi}}(\mathbf{z}_{t}|\mathbf{z}_{t-1})}\right]
\end{aligned}
$$

가 나온다.

---
---

# [*VDM*](#table-of-contents)
---
---

## **Variational Diffusion Models**

> VDM = MHVAE + 3 restrictions

1. The latent dimension is exactly equal to the data dimension
2. The structure of the latent encoder at each timestep is not learned; it is pre-defined as a **linear** Gaussian
model. In other words, it is a Gaussian distribution centered around the output of the previous timestep
    - linear 조건이 반드시 있어야 한다.(이유는 뒤에 설명) 
3. The Gaussian parameters of the latent encoders vary over time in such a way that the distribution of
the latent at final timestep T is a standard Gaussian

제한조건 1에서 관측데이터 $x$와 latent $z_{1:T}$들의 차원 수가 같으므로 time step $t \in [0,T]$에 대해, 관측데이터는 $t=0$일 때 $x_0$으로 두고, latent variables들은 $x_t,\ \text{where}\ t \in [1,T]$로 둘 수 있게 된다. 이 조건으로 인해 HVAE의 posterior Eq. 24가 VDM에서는 

**Equation 30:**  
$$q(\mathbf{x}_{1:T}|\mathbf{x}_0) = \prod_{t = 1}^{T}q(\mathbf{x}_{t}|\mathbf{x}_{t-1})$$

로 바뀌고,


제한조건 2에서 forward(encoding) process의 각 단계 $q(\mathbf{x}_{t}|\mathbf{x}_{t-1})$가 신경망 모델이 아니라 미리 설정된 linear 가우시안으로 두므로, t단계의 encoder는

**Equation 31:**
$$q(\mathbf{x}_{t}|\mathbf{x}_{t-1}) = \mathcal{N}(\mathbf{x}_{t} ; \sqrt{\alpha_t} \mathbf{x}_{t-1}, (1 - \alpha_t) \mathbf{I})$$

형태가 된다.

> 평균의 계수를 $\sqrt{\alpha_t}$, 분산의 계수를  $(1 - \alpha_t)$로 특이하게 두는 이유는, 각 단계의 분산이 1로 고정(variance-preserving)되기 때문.
> - 노이즈가 추가되는 각 단계에서 이전 단계의 그림 $x_{t-1}$을 그대로 두고 노이즈가 계속 추가되면 분산이 계속 증가하게 된다.
>   - $x_t$가 서로 독립인 $x_{t-1}$과 $\epsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$에 대해  $x_t = x_{t-1} + c\epsilon$의 형태이면 $x_{t-1}$과 $\epsilon$이 서로 독립이므로 분산의 성질 $V(aX + bY) = a^2V(X) + b^2V(Y)$을 적용해서 $x_t$의 분산을 구하면, $V(x_t) = V(x_{t-1}) + c^2 V(\epsilon) = V(x_{t-1}) + c^2$이 되서 양수가 계속 더해져서 분산이 증가하는 것을 확인 가능  
> - 논문과 동일하게 계수를 설정하면 분산이 1로 고정된다.
>   - $x_t = \sqrt{\alpha_t}x_{t-1} + \sqrt{1-\alpha_t}\epsilon$로 두면  
$V(x_t) = V(\sqrt{\alpha_t}x_{t-1}) + V(\sqrt{1-\alpha_t}\epsilon) = (\sqrt{\alpha_t})^2 V(x_{t-1}) + (\sqrt{1-\alpha_t})^2 V(\epsilon) = \alpha_t V(x_{t-1}) + (1-\alpha_t) V(\epsilon)$ 이 된다.  
$x_{t-1}$의 분산도 $\epsilon$과 동일하게 1이라면, $V(x_t) = \alpha_t (\mathbf{I}) + (1-\alpha_t) (\mathbf{I}) = (\alpha_t + 1 - \alpha_t) \mathbf{I} = \mathbf{I}$로 분산이 $\mathbf{I}$, 즉 1로 고정되는 것을 확인 가능  
>  
>  
>> 평균의 계수를 $\sqrt{\alpha_t}$, 분산의 계수를  $(1 - \alpha_t)$로 두어 Variance-Preserving을 설정하는 이유 
>> - 신경망 학습의 안정화
>>   - 딥러닝 모델은 입력값의 스케일이 일정할 때 가장 빠르고 안정적으로 학습한다. 스텝 $t$가 1이든 1,000이든 U-Net에 들어가는 이미지의 픽셀 분산이 항상 1 근처로 유지되므로, 모델이 극단적인 숫자에 압도되지 않는다.
>> - 목표 분포(Prior $x_T$)로의 정확한 수렴(제한 조건 3)
>>   - 디퓨전 모델의 forward process 최종 목표는 원본 이미지를 완벽한 순수 노이즈 $\mathcal{N}(\mathbf{0}, \mathbf{I})$로 만드는 것. 분산이 항상 $\mathbf{I}$로 묶여 있어야만 마지막 스텝에서 깔끔하게 표준 정규 분포에 도달 가능.
>> - 신호 대 잡음비(SNR)의 조절 
>>   - $\alpha_t$ 값은 시간이 지날수록 점점 작아지도록 설정한다. 즉, $\sqrt{\alpha_t}$(원본 신호의 비중)는 서서히 줄어들고 $\sqrt{1-\alpha_t}$(노이즈의 비중)는 서서히 커지게 된다($x_t = \sqrt{\alpha_t}x_{t-1} + \sqrt{1-\alpha_t}\epsilon$). 분산을 1로 유지하면서 데이터와 노이즈의 섞이는 비율을 조절할 수 있다.

제한 조건 3을 만족하도록 forward process를 설정하므로 $p(x_T)$가 표준 정규분포가 된다. 제한 조건1과 3을 이용하면 HVAE의 joint distribution Eq. 23가

**Equation 32, 33:**
$$
\begin{aligned}
p(\mathbf{x}_{0:T}) &= p(\mathbf{x}_T)\prod_{t=1}^{T}p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t) \\
\text{where,} \\
p(\mathbf{x}_T) &= \mathcal{N}(\mathbf{x}_T; \mathbf{0}, \mathbf{I})
\end{aligned}
$$
이 된다.

이 세가지 제한 조건을 만족하는 VDM을 이미지로 표현하면 아래와 같다.

![VDM](imgs/vdm_base.png)

[이미지 출처](https://arxiv.org/abs/2208.11970)

## **VDM ELBO 유도 1**

**Equation 34~45:**

$$
\begin{aligned}
\log p(\mathbf{x})
&= \log \int p(\mathbf{x}_{0:T}) d\mathbf{x}_{1:T}\\
&= \log \int \frac{p(\mathbf{x}_{0:T})q(\mathbf{x}_{1:T}|\mathbf{x}_0)}{q(\mathbf{x}_{1:T}|\mathbf{x}_0)} d\mathbf{x}_{1:T}\\
&= \log \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\frac{p(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\right]\\
&\geq \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\right] \\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)\prod_{t=1}^{T}p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{\prod_{t = 1}^{T}q(\mathbf{x}_{t}|\mathbf{x}_{t-1})}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)\prod_{t=2}^{T}p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_T|\mathbf{x}_{T-1})\prod_{t = 1}^{T-1}q(\mathbf{x}_{t}|\mathbf{x}_{t-1})}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)\prod_{t=1}^{T-1}p_{\boldsymbol{\theta}}(\mathbf{x}_{t}|\mathbf{x}_{t+1})}{q(\mathbf{x}_T|\mathbf{x}_{T-1})\prod_{t = 1}^{T-1}q(\mathbf{x}_{t}|\mathbf{x}_{t-1})}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)}{q(\mathbf{x}_T|\mathbf{x}_{T-1})}\right] + \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \prod_{t = 1}^{T-1}\frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t}|\mathbf{x}_{t+1})}{q(\mathbf{x}_{t}|\mathbf{x}_{t-1})}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)\right] + \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)}{q(\mathbf{x}_T|\mathbf{x}_{T-1})}\right] + \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[ \sum_{t=1}^{T-1} \log \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t}|\mathbf{x}_{t+1})}{q(\mathbf{x}_{t}|\mathbf{x}_{t-1})}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)\right] + \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)}{q(\mathbf{x}_T|\mathbf{x}_{T-1})}\right] + \sum_{t=1}^{T-1}\mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[ \log \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t}|\mathbf{x}_{t+1})}{q(\mathbf{x}_{t}|\mathbf{x}_{t-1})}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1}|\mathbf{x}_0)}\left[\log p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)\right] + \mathbb{E}_{q(\mathbf{x}_{T-1}, \mathbf{x}_T|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)}{q(\mathbf{x}_T|\mathbf{x}_{T-1})}\right] + \sum_{t=1}^{T-1}\mathbb{E}_{q(\mathbf{x}_{t-1}, \mathbf{x}_t, \mathbf{x}_{t+1}|\mathbf{x}_0)}\left[\log \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t}|\mathbf{x}_{t+1})}{q(\mathbf{x}_{t}|\mathbf{x}_{t-1})}\right]\\
&=  \begin{aligned}[t]
      \underbrace{\mathbb{E}_{q(\mathbf{x}_{1}|\mathbf{x}_0)}\left[\log p_{\theta}(\mathbf{x}_0|\mathbf{x}_1)\right]}_\text{reconstruction term} &- \underbrace{\mathbb{E}_{q(\mathbf{x}_{T-1}|\mathbf{x}_0)}\left[D_{\text{KL}}(q(\mathbf{x}_T|\mathbf{x}_{T-1}) \| p(\mathbf{x}_T))\right]}_\text{prior matching term} \\
      &- \sum_{t=1}^{T-1}\underbrace{\mathbb{E}_{q(\mathbf{x}_{t-1}, \mathbf{x}_{t+1}|\mathbf{x}_0)}\left[D_{\text{KL}}(q(\mathbf{x}_{t}|\mathbf{x}_{t-1}) \| p_{\theta}(\mathbf{x}_{t}|\mathbf{x}_{t+1}))\right]}_\text{consistency term}
    \end{aligned}
\end{aligned}
$$

## **VDM ELBO 유도 2**

**Equation 46:**

$$
\begin{aligned}
q(\mathbf{x}_t | \mathbf{x}_{t-1}, \mathbf{x}_0) = \frac{q(\mathbf{x}_{t-1}|\mathbf{x}_t, \mathbf{x}_0)q(\mathbf{x}_t|\mathbf{x}_0)}{q(\mathbf{x}_{t-1}|\mathbf{x}_0)}
\end{aligned}
$$


**Equation 47~58:**

$$
\begin{aligned}
\log p(\mathbf{x})
&\geq \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)\prod_{t=1}^{T}p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{\prod_{t = 1}^{T}q(\mathbf{x}_{t}|\mathbf{x}_{t-1})}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)\prod_{t=2}^{T}p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_1|\mathbf{x}_0)\prod_{t = 2}^{T}q(\mathbf{x}_{t}|\mathbf{x}_{t-1})}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)\prod_{t=2}^{T}p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_1|\mathbf{x}_0)\prod_{t = 2}^{T}q(\mathbf{x}_{t}|\mathbf{x}_{t-1}, \mathbf{x}_0)}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_T)p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)}{q(\mathbf{x}_1|\mathbf{x}_0)} + \log \prod_{t=2}^{T}\frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_{t}|\mathbf{x}_{t-1}, \mathbf{x}_0)}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)}{q(\mathbf{x}_1|\mathbf{x}_0)} + \log \prod_{t=2}^{T}\frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{\frac{q(\mathbf{x}_{t-1}|\mathbf{x}_{t}, \mathbf{x}_0)q(\mathbf{x}_t|\mathbf{x}_0)}{q(\mathbf{x}_{t-1}|\mathbf{x}_0)}}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)}{q(\mathbf{x}_1|\mathbf{x}_0)} + \log \prod_{t=2}^{T}\frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{\frac{q(\mathbf{x}_{t-1}|\mathbf{x}_{t}, \mathbf{x}_0)\cancel{q(\mathbf{x}_t|\mathbf{x}_0)}}{\cancel{q(\mathbf{x}_{t-1}|\mathbf{x}_0)}}}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)}{\cancel{q(\mathbf{x}_1|\mathbf{x}_0)}} + \log \frac{\cancel{q(\mathbf{x}_1|\mathbf{x}_0)}}{q(\mathbf{x}_T|\mathbf{x}_0)} + \log \prod_{t=2}^{T}\frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_{t-1}|\mathbf{x}_{t}, \mathbf{x}_0)}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)}{q(\mathbf{x}_T|\mathbf{x}_0)} +  \sum_{t=2}^{T}\log\frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_{t-1}|\mathbf{x}_{t}, \mathbf{x}_0)}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)\right] + \mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)}{q(\mathbf{x}_T|\mathbf{x}_0)}\right] + \sum_{t=2}^{T}\mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}\left[\log\frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_{t-1}|\mathbf{x}_{t}, \mathbf{x}_0)}\right]\\
&= \mathbb{E}_{q(\mathbf{x}_{1}|\mathbf{x}_0)}\left[\log p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)\right] + \mathbb{E}_{q(\mathbf{x}_{T}|\mathbf{x}_0)}\left[\log \frac{p(\mathbf{x}_T)}{q(\mathbf{x}_T|\mathbf{x}_0)}\right] + \sum_{t=2}^{T}\mathbb{E}_{q(\mathbf{x}_{t}, \mathbf{x}_{t-1}|\mathbf{x}_0)}\left[\log\frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_{t-1}|\mathbf{x}_{t}, \mathbf{x}_0)}\right]\\
&= \underbrace{\mathbb{E}_{q(\mathbf{x}_{1}|\mathbf{x}_0)}\left[\log p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)\right]}_\text{reconstruction term} - \underbrace{D_{\text{KL}}(q(\mathbf{x}_T|\mathbf{x}_0) \| p(\mathbf{x}_T))}_\text{prior matching term} - \sum_{t=2}^{T} \underbrace{\mathbb{E}_{q(\mathbf{x}_{t}|\mathbf{x}_0)}\left[D_{\text{KL}}(q(\mathbf{x}_{t-1}|\mathbf{x}_t, \mathbf{x}_0) \| p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t))\right]}_\text{denoising matching term}
\end{aligned}
$$

---
---
---

# [document tail](#table-of-contents)
***
***
***
***